#### Keras를 이용한 손글씨 인식 딥러닝
- TensorFlow로 작성된 프로그램은 sklearn보다 무지하게 어렵다.</br>
-> Keras = TensorFlow 에 포함됨. 고수준 연산은 TensorFlow 를 사용한다.
- Keras는 테아노(Theano)와 TensorFlow를 Wrapping한 라이브러리

#### Data import

In [53]:
import pandas as pd

train = pd.read_csv('../Data/train_20k.csv')
train.head()

,5,0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,...,0.608,0.609,0.610,0.611,0.612,0.613,0.614,0.615,0.616,0.617
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [54]:
test = pd.read_csv('../Data/test_1k.csv')
test.head()

,7,0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,...,0.658,0.659,0.660,0.661,0.662,0.663,0.664,0.665,0.666,0.667
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


#### 전처리(Preprocessing)
##### 1. 결측치 확인

In [4]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Columns: 785 entries, 5 to 0.617
dtypes: int64(785)
memory usage: 119.8 MB


In [8]:
# train 의 결측치
train.isnull().sum().sum()
# -> 이런 형태를 함수적 언어라  한다

np.int64(0)

In [11]:
test.isnull().sum().sum()

np.int64(0)

##### 2. Target Data의 비율 확인

In [41]:
import numpy as np

In [ ]:
# import numpy as np

# # 1. train 변수 자체가 존재하는지, None은 아닌지 검사
# if 'train' in locals() and train is not None:
#     try:
#         # 2. train[0] 대신 안전하게 전체 데이터를 넘파이 배열로 변환 시도
#         arr = np.asarray(train)
        
#         # 3. 배열이 비어있지 않고 크기가 존재하는지 확인 (넘파이 전용 안전 함수)
#         if arr.size > 0:
#             # 4. 첫 번째 데이터(train[0] 역할)를 안전하게 슬라이싱
#             first_element = arr[0]
            
#             # 5. 1차원으로 펼치고 고유값 개수 추출
#             target_data = np.ravel(first_element)
#             values, counts = np.unique(target_data, return_counts=True)
#             checkCount = counts
#             print("성공! counts 결과:", checkCount)
#         else:
#             print("train 배열의 크기가 0(비어 있음)입니다.")
#             checkCount = None
            
#     except Exception as e:
#         print(f"데이터 구조 문제로 처리에 실패했습니다. 에러: {e}")
#         checkCount = None
# else:
#     print("train 변수가 정의되지 않았거나 None입니다.")


성공! counts 결과: [609   1   3   1   2   1   1   2   2   1   1   1   1   4   2   2   1   1
   2   1   1   2   1   1   1   2   6   1   1   1   2   1   1   1   1   1
   1   1   2   1   1   2   1   1   3   2   1   1   2   1   1   1   2   2
   2   1   1   1   1   1   2   1   3   1   1   1   1   3   1   4   1   1
   1   1   1  41  23   2]


In [ ]:
# 1. train을 넘파이가 다룰 수 있는 깨끗한 형태(Object 배열 등)로 먼저 변환합니다.
# train = np.array(train, dtype=float)

# # 2. 원래 작성하셨던 코드를 그대로 실행합니다. (이제 에러 없이 정상 작동합니다!)
# checkCount = np.unique(train[0], return_counts=True)[1]

# np.min(checkCount) / np.max(checkCount)

> Data의 Target 비율이 최소 77% 가 넘는다

#### train 과 test 를 Target과 Feature로 분리하고 정규화 하기

In [60]:
# Train의 Data 와 Target 분리
train_label = train.loc[:, train.columns == 0]
train_data = train.loc[:, train.columns != 0] / 255.0

# Train의 Data 와 Target 분리
test_label = test.loc[:, test.columns == 0]
test_data = test.loc[:, test.columns != 0] / 255.0

> 255로 나누는 이유 : 이미지 픽셀 데이터를 0.0 ~ 1.0 사이의 작은 실수 값으로 바꾸는 '정규화(Normalization)' 작업을 하기 위해서입니다</br>
픽셀 값의 범위가 0~255이기 때문

---
#### Deep Learning Model 만들기

In [67]:
from tensorflow import keras
from tensorflow.keras.layers import Input

In [68]:
# 0번 열을 제외한 784개의 픽셀 데이터만 골라내고 정규화합니다.
train_data = train.iloc[:, 1:] / 255.0

# 0번 열은 정답(레이블)으로 따로 저장합니다.
train_label = train.iloc[:, 0]

In [69]:
# 모델 생성
model = keras.Sequential()

model.add(Input(shape=(784,))) # 입력층
model.add(keras.layers.Dense(100, activation='relu')) # 은닉층. 은닉층은 여러개 가능 = 100 이외에 임의의 숫자 적용 가능
model.add(keras.layers.Dense(10, activation='softmax')) # 출력층. 입력층과 출력층의 개수는 정해져 있다.

# 손실함수
model.compile(
    optimizer = 'adam',
    loss = 'sparse_categorical_crossentropy',
    metrics = ['accuracy']
)

# 모델 훈련
model.fit(
    np.array(train_data),
    np.array(train_label),
    epochs = 10
)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8829 - loss: 0.4174
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9426 - loss: 0.2034
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9589 - loss: 0.1467
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9674 - loss: 0.1134
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9743 - loss: 0.0911
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9798 - loss: 0.0695
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9844 - loss: 0.0555
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9872 - loss: 0.0463
Epoch 9/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9900 - loss: 0.0365
Epoch 10/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9923 - loss: 0.0299


#### 테스트 데이터로 평가하기

In [70]:
# 1. test 데이터도 첫 번째 열(정답)과 나머지 열(픽셀)로 안전하게 분리합니다.
test_data = test.iloc[:, 1:] / 255.0  # 두 번째 열부터 끝까지 (784개)
test_label = test.iloc[:, 0]           # 첫 번째 열 (정답 레이블)

# 2. 평가 결과를 'score' 변수에 저장합니다. (기존 코드에 score = 를 추가)
score = model.evaluate(test_data, test_label)

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9670 - loss: 0.1095  


In [71]:
model.evaluate(test_data, test_label)
print("loss =", score[0])
print("accuracy =", score[1])

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9670 - loss: 0.1095 
loss = 0.10953761637210846
accuracy = 0.9670000076293945


In [72]:
model.evaluate(train_data, train_label)
print("loss =", score[0])
print("accuracy =", score[1])

625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 905us/step - accuracy: 0.9966 - loss: 0.0201
loss = 0.10953761637210846
accuracy = 0.9670000076293945


---
#### test_data로 predict 해보기

In [73]:
pred = model.predict(test_data)
print("test label", test_label[:10])
print("pred", np.argmax(pred[:10], axis=1)) # argmax : 예측값중 최대값의 index

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
test label 0    2
1    1
2    0
3    4
4    1
5    4
6    9
7    5
8    9
9    0
Name: 7, dtype: int64
pred [2 1 0 4 1 4 9 2 9 0]


In [74]:
# test_data의 0번 예측값 확인
print("정답 : ", test_label.loc[0])
print("예측값 : ", np.argmax(pred[0]))

정답 :  2
예측값 :  2
